# Agents + MCP — hands-on

**Goal:** see *why* the Model Context Protocol exists by building the same flight-booking agent twice: once without MCP, once with it.

| Part | What you'll do |
|---|---|
| 1 | Meet the problem: three airlines, three different APIs |
| 2 | **Before MCP** — one hand-written tool per airline |
| 3–4 | **The MCP server** — one interface in front of all three airlines; call it with *no LLM* |
| 5 | **With MCP** — the agent discovers its tools at runtime and asks a human before booking |
| 6 | Tools come from *config*, not code |
| 7 | Exercise: add a fourth airline without touching the agent |
| 8 | The same server over **HTTP** — what lets a remote agent use it |
| 9 | Next: connect Hermes on Railway |

```text
BEFORE MCP                                    WITH MCP
Agent ── joyair_search  ──► Joy Air API       Agent (MCP client) ──MCP──► Flight MCP server ──► Joy Air API
      ── draair_search  ──► Dra Air API                                                    ├──► Dra Air API
      ── aeroggo_search ──► Aeroggo API                                                    └──► Aeroggo API
 new airline = new tool + new agent code       new airline = a server change only
```

**You need:** Python 3.10+ and an [OpenRouter](https://openrouter.ai) API key in your `.env` file (copy `.env.example` to `.env` and paste the key; Claude is reached through OpenRouter). The first cell installs the two packages (`anthropic`, `mcp`) into this notebook's own environment. Run the notebook from the project folder (the one with `mcp_config.json`).

In [1]:
import sys
from pathlib import Path

assert sys.version_info >= (3, 10), f"Python 3.10+ is required (this kernel runs {sys.version.split()[0]})"
assert Path("mcp_config.json").exists(), "Open this notebook from the project folder, next to mcp_config.json"

# Installs into THIS kernel's environment; a no-op when already satisfied. Pinned because mcp 2.x renamed FastMCP.
%pip install -q --disable-pip-version-check -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv

# llm.py builds the Claude client when it is imported, so the key must be in place first. It normally comes from
# your .env file (a real environment variable wins); otherwise you are asked here (kept in memory, never saved).
load_dotenv(".env")
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

import json
from contextlib import AsyncExitStack
from IPython.display import Code

import airlines, naive_agent, mcp_agent

TASK = "Find flights from SFO to JFK and book the cheapest one for Jane Tan."
print("Model:", mcp_agent.MODEL)

Model: anthropic/claude-sonnet-4.6


## 1 · The problem

The same question — *"flights from SFO to JFK?"* — asked of three airlines. Look at what changes: the **endpoint name**, the **input names** (`origin/destination` vs `src/dst` vs `start/finish`), and the **response shape**.

In [3]:
for label, result in [
    ("Joy Air  · GET /api/flights ", airlines.joyair_api_flights("SFO", "JFK")),
    ("Dra Air  · GET /flights-list", airlines.draair_flights_list("SFO", "JFK")),
    ("Aeroggo  · POST list_flights", airlines.aeroggo_list_flights("SFO", "JFK")),
]:
    print(label, "\n ", result, "\n")

Joy Air  · GET /api/flights  
  [{'flight_number': 'JA101', 'origin': 'SFO', 'destination': 'JFK', 'price_usd': 310}, {'flight_number': 'JA205', 'origin': 'SFO', 'destination': 'JFK', 'price_usd': 275}] 

Dra Air  · GET /flights-list 
  {'results': [{'flightNo': 'DR330', 'from': 'SFO', 'to': 'JFK', 'fare': {'amount': 289, 'currency': 'USD'}}]} 

Aeroggo  · POST list_flights 
  {'detailedFlights': [{'flight': 'AG78', 'start': 'SFO', 'finish': 'JFK', 'cost': 342.5}, {'flight': 'AG91', 'start': 'SFO', 'finish': 'JFK', 'cost': 259.0}]} 



## 2 · Before MCP: one hand-written tool per airline

`naive_agent.py` gives Claude one tool per airline, each with its own input names, and forwards each call to the matching function. Claude has to read three response shapes and work out that `price_usd`, `fare.amount` and `cost` all mean *price*.

Two limits to notice: there is **no booking tool** (the file says "…and 3 more book tools"), and every new airline means a new schema *and* new code inside the agent.

In [4]:
for t in naive_agent.TOOLS:
    print(f"{t['name']:16} inputs: {list(t['input_schema']['properties'])}")

joyair_search    inputs: ['origin', 'destination']
draair_search    inputs: ['src', 'dst']
aeroggo_search   inputs: ['start', 'finish']


In [5]:
print(naive_agent.run("Find me the cheapest flight from SFO to JFK across all airlines."))

Here's a summary of all available flights from **SFO → JFK**, sorted by price:

| Rank | Flight | Airline | Price (USD) |
|------|--------|---------|-------------|
| 🥇 1 | AG91 | Aeroggo | **$259.00** |
| 2 | JA205 | Joy Air | $275.00 |
| 3 | DR330 | Dra Air | $289.00 |
| 4 | JA101 | Joy Air | $310.00 |
| 5 | AG78 | Aeroggo | $342.50 |

### 🏆 Cheapest Flight: **Aeroggo Flight AG91 at $259.00**

That's the best deal across all airlines! Would you like to go ahead and book this flight, or need any more details?


## 3 · The MCP server: one interface in front of all three

`flight_mcp_server.py` hides the three APIs behind two tools, `search_flights` and `book_flight`. The `@mcp.tool()` decorator turns each function's signature and docstring into the tool schema Claude sees.

In [6]:
Code(filename="flight_mcp_server.py")

"""A Flight MCP server. It hides the three different airline APIs behind ONE standard interface.
Any MCP client (our agent, Claude Desktop, Cursor, Cline...) can discover and call these tools."""
import os
from mcp.server.fastmcp import FastMCP
import airlines

# Local default is stdio. On Railway set MCP_TRANSPORT=streamable-http and MCP_HOST=0.0.0.0 (it injects PORT).
mcp = FastMCP("flight-sim", host=os.environ.get("MCP_HOST", "127.0.0.1"), port=int(os.environ.get("PORT", 8000)))


@mcp.tool()
def search_flights(origin: str, destination: str) -> list[dict]:
    """Search Joy Air, Dra Air and Aeroggo for flights. Returns one normalised list."""
    results = []
    for f in airlines.joyair_api_flights(origin, destination):
        results.append({"airline": "Joy Air", "flight": f["flight_number"], "price_usd": f["price_usd"]})
    for f in airlines.draair_flights_list(origin, destination)["results"]:
        results.append({"airline": "Dra Air", "flight": f["flightNo"], "price_usd": f["fare"]["amount"]})
    for f in airlines.aeroggo_list_flights(origin, destination)["detailedFlights"]:
        results.append({"airline": "Aeroggo", "flight": f["flight"], "price_usd": f["cost"]})
    return sorted(results, key=lambda r: r["price_usd"])


@mcp.tool()
def book_flight(airline: str, flight: str, passenger_name: str) -> dict:
    """Book a flight. Needs the airline and flight from search_flights, plus the passenger's full name."""
    if airline == "Joy Air":
        ref = airlines.joyair_api_book(flight, passenger_name)["booking_ref"]
    elif airline == "Dra Air":
        ref = airlines.draair_reserve(flight, passenger_name)["reservation"]["code"]
    elif airline == "Aeroggo":
        ref = airlines.aeroggo_book(flight, passenger_name)["confirmation"]
    else:
        return {"error": f"Unknown airline {airline!r}"}
    return {"status": "booked", "booking_reference": ref, "passenger": passenger_name}


if __name__ == "__main__":
    # stdio: the client launches this script and talks to it. streamable-http: the client connects to a URL.
    mcp.run(transport=os.environ.get("MCP_TRANSPORT", "stdio"))

## 4 · Talk to the server with no LLM

MCP is just a protocol: **list the tools**, then **call a tool**. `connect_servers` reads `mcp_config.json`, launches each server as a subprocess (the *stdio* transport) and asks it what tools it has.

In [16]:
async with AsyncExitStack() as stack:
    sessions, tools = await mcp_agent.connect_servers(stack)
    for t in tools:
        print(f"TOOL: {t['name']}  inputs: {list(t['input_schema']['properties'])}")

    res = await sessions["search_flights"].call_tool("search_flights", {"origin": "SFO", "destination": "JFK"})
    print("\nSEARCH RESULT (cheapest first):")
    for c in res.content:
        print(" ", c.text.replace("\n", " "))

print("\nThis JSON is everything Claude ever sees about a tool:")
print(json.dumps(tools[0], indent=2))

TOOL: search_flights  inputs: ['origin', 'destination']
TOOL: book_flight  inputs: ['airline', 'flight', 'passenger_name']

SEARCH RESULT (cheapest first):
  {   "airline": "Aeroggo",   "flight": "AG91",   "price_usd": 259.0 }
  {   "airline": "Joy Air",   "flight": "JA205",   "price_usd": 275 }
  {   "airline": "Dra Air",   "flight": "DR330",   "price_usd": 289 }
  {   "airline": "Joy Air",   "flight": "JA101",   "price_usd": 310 }
  {   "airline": "Aeroggo",   "flight": "AG78",   "price_usd": 342.5 }

This JSON is everything Claude ever sees about a tool:
{
  "name": "search_flights",
  "description": "Search Joy Air, Dra Air and Aeroggo for flights. Returns one normalised list.",
  "input_schema": {
    "properties": {
      "origin": {
        "title": "Origin",
        "type": "string"
      },
      "destination": {
        "title": "Destination",
        "type": "string"
      }
    },
    "required": [
      "origin",
      "destination"
    ],
    "title": "search_flightsArgum

## 5 · With MCP: the agent discovers its tools

`mcp_agent.run_agent` contains no airline code. It connects to the servers in the config, copies each discovered tool schema straight into the Claude request, and routes every `tool_use` back to the server that owns it.

`book_flight` is in `NEEDS_APPROVAL`, so a human has to type `y` — an input box appears in the notebook when Claude tries to book.

In [22]:
print(await mcp_agent.run_agent(TASK))

Discovered tools: ['search_flights', 'book_flight']
[step 0] search_flights({'origin': 'SFO', 'destination': 'JFK'})
[step 1] book_flight({'airline': 'Aeroggo', 'flight': 'AG91', 'passenger_name': 'Jane Tan'})
All done! Here's a summary:

- **Flight:** Aeroggo AG91
- **Route:** SFO → JFK
- **Price:** $259.00 (cheapest available)
- **Passenger:** Jane Tan
- **Booking Reference:** AGO-AG91-1188

Jane is all set to fly! ✈️


## 6 · Tools come from config, not code

Same agent, same task, but `empty_config.json` lists no servers. Nothing is discovered, so Claude has no tools and can only say it can't help. Swap the config, swap the capabilities.

In [23]:
print(await mcp_agent.run_agent(TASK, "empty_config.json"))

Discovered tools: []
I'd be happy to help book a flight for Jane Tan, but I need to be straightforward with you:

**I can't actually perform this task** because I:

1. **Don't have access to flight search tools** – I cannot query real-time flight databases (like Google Flights, Expedia, Airlines.com, etc.)
2. **Can't make bookings** – I have no ability to access booking systems, process payments, or create reservations.
3. **Don't have current pricing data** – My knowledge has a cutoff and I don't have live fare information.

---

## Here's How You Can Do This:

### 🔍 Search for Flights (SFO → JFK):
- **Google Flights** – [flights.google.com](https://flights.google.com)
- **Kayak** – [kayak.com](https://kayak.com)
- **Expedia** – [expedia.com](https://expedia.com)
- **Skyscanner** – [skyscanner.com](https://skyscanner.com)

### ✈️ Book Directly with Airlines:
- **Delta**, **United**, **JetBlue**, and **American** all fly SFO–JFK routes frequently.

### 💡 Tips for Finding the Cheapest F

## 7 · Exercise: add a fourth airline

1. In `airlines.py`, add a fourth fake airline with its *own* odd naming, e.g. `skybee_find(leaving_from, going_to)` returning `{"trips": [{"id": "SB12", "usd_cents": 24900}]}`, plus a booking function.
2. In `flight_mcp_server.py`, normalise it into `search_flights` (mind the cents!) and add a branch to `book_flight`.
3. Re-run Part 5 **unchanged**.

The agent picks the new airline up with no agent-side change. Compare Part 2, where you would add a schema, a `FUNCS` entry and a booking tool. (Stdio servers are launched fresh on every `run_agent`, so no kernel restart is needed for server edits.)

## 8 · Same server, over the network

So far the agent *launched* the server on your machine (stdio). A hosted agent, like Hermes on Railway, can't do that: it has to *dial a URL*. MCP's answer is the **streamable HTTP** transport. The server needs only a switch, `MCP_TRANSPORT=streamable-http` (`MCP_HOST` and `PORT` control where it listens), and `http_config.json` uses the same `url` key that Hermes uses in its own config.

In [18]:
import atexit, socket, subprocess, sys, time
from urllib.parse import urlparse

url = json.load(open("http_config.json"))["mcpServers"]["flight-sim"]["url"]
port = urlparse(url).port          # the config is the single source of truth for where the server listens

def port_open():
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

assert not port_open(), f"Port {port} is already in use: pick another one in http_config.json"
server = subprocess.Popen([sys.executable, "flight_mcp_server.py"],
                          env={**os.environ, "MCP_TRANSPORT": "streamable-http", "PORT": str(port)})
atexit.register(server.terminate)

for _ in range(100):               # wait until it accepts connections
    if port_open():
        break
    assert server.poll() is None, "The server exited during startup"
    time.sleep(0.1)
print("Server up at", url)

Server up at http://127.0.0.1:8765/mcp


INFO:     Started server process [65800]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)


In [19]:
async with AsyncExitStack() as stack:
    sessions, tools = await mcp_agent.connect_servers(stack, "http_config.json")
    print("Discovered over HTTP:", [t["name"] for t in tools])

INFO:     127.0.0.1:50679 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50680 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50681 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50682 - "POST /mcp HTTP/1.1" 200 OK
Discovered over HTTP: ['search_flights', 'book_flight']
INFO:     127.0.0.1:50683 - "DELETE /mcp HTTP/1.1" 200 OK


Created new transport with session ID: fa4c48f52fa6403481305d0ed267e32b
Processing request of type ListToolsRequest
Terminating session: fa4c48f52fa6403481305d0ed267e32b


In [20]:
print(await mcp_agent.run_agent(TASK, "http_config.json"))

Created new transport with session ID: 2ad242ceb42b4c76b35e7952c579a928
Processing request of type ListToolsRequest


INFO:     127.0.0.1:50684 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50685 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50686 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50687 - "POST /mcp HTTP/1.1" 200 OK
Discovered tools: ['search_flights', 'book_flight']
[step 0] search_flights({'origin': 'SFO', 'destination': 'JFK'})
INFO:     127.0.0.1:50690 - "POST /mcp HTTP/1.1" 200 OK


Processing request of type CallToolRequest


[step 1] book_flight({'airline': 'SkyBee', 'flight': 'SB12', 'passenger_name': 'Jane Tan'})
INFO:     127.0.0.1:50717 - "POST /mcp HTTP/1.1" 200 OK


Processing request of type CallToolRequest


INFO:     127.0.0.1:50720 - "DELETE /mcp HTTP/1.1" 200 OK
The booking is confirmed! Here's a summary:

- **Flight:** SkyBee SB12
- **Route:** SFO → JFK
- **Price:** $249.00 (cheapest available)
- **Passenger:** Jane Tan
- **Booking Reference:** SKY-SB12-5502


Terminating session: 2ad242ceb42b4c76b35e7952c579a928


In [21]:
server.terminate()
server.wait()
print("Server stopped.")

INFO:     Shutting down
INFO:     Waiting for application shutdown.
StreamableHTTP session manager shutting down
INFO:     Application shutdown complete.
INFO:     Finished server process [65800]


Server stopped.


## 9 · Next: connect Hermes on Railway

> **Status: not run yet.** The config below follows the [Hermes MCP docs](https://hermes-agent.nousresearch.com/docs/user-guide/features/mcp); we'll validate it once the Railway services are up.

Hermes takes MCP servers by URL (plain `http://` is only accepted for localhost), so `127.0.0.1:8000` on this laptop can't be reached from Railway. The plan:

1. **Deploy this server as a Railway service** with `MCP_TRANSPORT=streamable-http` and `MCP_HOST=0.0.0.0` (Railway supplies `PORT`), start command `python flight_mcp_server.py`. The endpoint is `https://<service>.up.railway.app/mcp`.
   `MCP_HOST=0.0.0.0` matters: with the default `127.0.0.1` the MCP SDK answers any non-localhost `Host` header with HTTP 421 (checked locally).
2. **Add it to Hermes** in `~/.hermes/config.yaml` (on Railway that folder must be on a persistent volume):
   ```yaml
   mcp_servers:
     flight_sim:
       url: "https://<service>.up.railway.app/mcp"
       tools:
         exclude: [book_flight]     # optional: keep the demo read-only
   ```
3. **Check it:** `hermes mcp test flight_sim`, then `/reload-mcp` in chat. Hermes exposes the tools as `mcp_flight_sim_search_flights` and `mcp_flight_sim_book_flight`.
4. **Before the URL goes public:** this server has no authentication and `book_flight` has side effects. Add a bearer-token check to the server and pass it with `headers: {Authorization: "Bearer …"}`; both Hermes and `mcp_agent.connect_servers` accept `headers`.